# GPU Debate Backfill

Jupyter에서 과거 토론 백필을 실행하고, 진행률 확인과 실패 복구까지 처리합니다.

사전 준비:
- EC2 `3_ai_server` 접근 가능
- `.env` 설정 완료
- vLLM 또는 OpenAI 호환 LLM 서버 실행 완료

## 0. vLLM 권장 실행 예시

```bash
vllm serve Qwen/Qwen2.5-7B-Instruct \
  --dtype auto \
  --max-model-len 4096 \
  --gpu-memory-utilization 0.6 \
  --disable-log-requests
```

In [ ]:
import os
import sys
from datetime import date, datetime
from pathlib import Path

PROJECT_DIR = Path("/home/ssafy/dev-ai/services/ai/4_debate_worker")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f"CWD: {Path.cwd()}")

In [ ]:
from core.config import settings
from core.logger import add_file_handler
from worker.backfill_runner import BackfillOptions
from worker.runtime import build_runtime

runtime = build_runtime()
log_path = settings.log_dir / f"notebook_backfill_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
add_file_handler(log_path)
print(f"log_path={log_path}")

In [ ]:
snapshot = runtime.resource_monitor.capture()
recommended_gpu_util = runtime.resource_monitor.recommend_vllm_gpu_memory_utilization()
snapshot, recommended_gpu_util

In [ ]:
START_DATE = date(2025, 1, 1)
END_DATE = date.today()
SOURCE = "ml_reports"   # 필요 시 "trading_history"
PORTFOLIO_ID = None      # trading_history일 때만 숫자 지정
MAX_TARGETS = 200

options = BackfillOptions(
    start_date=START_DATE,
    end_date=END_DATE,
    source=SOURCE,
    market_type=settings.TARGET_MARKET_TYPE,
    portfolio_id=PORTFOLIO_ID,
    max_targets=MAX_TARGETS,
    debate_version=settings.DEBATE_VERSION,
    news_limit=settings.NEWS_LIMIT,
    financial_limit=settings.FINANCIAL_LIMIT,
    lease_seconds=settings.JOB_LEASE_SECONDS,
    max_retry_attempts=settings.MAX_RETRY_ATTEMPTS,
    retry_backoff_seconds=settings.RETRY_BACKOFF_SECONDS,
    poll_interval_seconds=settings.BACKFILL_POLL_INTERVAL_SECONDS,
)
options

In [ ]:
summary = runtime.backfill_runner.run(options)
summary

In [ ]:
summary = runtime.backfill_runner.report(options)
daily_statuses = runtime.backfill_runner.get_daily_statuses(options)
summary, daily_statuses[:5], daily_statuses[-5:]

In [ ]:
FAILED_DATE = date(2025, 1, 3)
failed_jobs = runtime.backfill_runner.list_failed_jobs(
    report_date=FAILED_DATE,
    source=SOURCE,
    portfolio_id=PORTFOLIO_ID,
    limit=20,
)
failed_jobs

In [ ]:
# 저장 API만 다시 시도하려면 clear_result_payload=False
# LLM 추론부터 다시 하려면 clear_result_payload=True
UPDATED = runtime.backfill_runner.requeue_failed_jobs(
    report_date=FAILED_DATE,
    source=SOURCE,
    portfolio_id=PORTFOLIO_ID,
    statuses=("failed_permanent", "failed_retryable"),
    clear_result_payload=False,
)
UPDATED